# Polarized initial state

```{autolink-concat}

```

The other example notebooks formulate the **unpolarized** intensity: an incoherent sum over the helicity of the initial state, which is what one measures after integrating over the orientation of the three-body decay plane. If the initial state is produced *polarized*, the intensity also depends on three Euler angles $(\phi, \theta, \chi)$ that describe this orientation with respect to the production frame, in combination with a **spin-density matrix** $\rho$ {cite}`JPAC:2019ufm`. See [this section](https://redeboer.github.io/phd-thesis/chapter3.html#sec-differential-decay-rate) for the theory.

This notebook formulates such a polarized intensity for $J/\psi \to K^0 \Sigma^+ \overline{p}$, where the $J/\psi$ originates from an $e^+e^-$ collision, and compares the resulting distributions with the unpolarized case.

In [ ]:
import logging
import os
import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import qrules
import sympy as sp
from IPython.display import Latex, Markdown
from matplotlib.colors import CenteredNorm
from tensorwaves.data.transform import SympyDataTransformer

from ampform_dpd import DalitzPlotDecompositionBuilder
from ampform_dpd.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform_dpd.dynamics.builder import formulate_breit_wigner_with_form_factor
from ampform_dpd.io import (
    as_markdown_table,
    aslatex,
    cached,
    mute_ampform_warnings,
    simplify_latex_rendering,
)
from ampform_dpd.polarization import (
    create_spin_density_matrix,
    formulate_ee_annihilation_density,
    formulate_polarized_spin_half_density,
    formulate_unpolarized_density,
)

simplify_latex_rendering()
logging.getLogger("absl").setLevel(logging.ERROR)  # mute JAX
warnings.simplefilter("ignore", category=RuntimeWarning)

if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Decay definition

We take the same decay as in {doc}`jpsi2ksp`, but limit the number of resonances to two, in order to keep the expressions small.

In [ ]:
REACTION = qrules.generate_transitions(
    initial_state=[("J/psi(1S)", [+1])],
    final_state=["K0", ("Sigma+", [+0.5]), ("p~", [+0.5])],
    allowed_interaction_types="strong",
    allowed_intermediate_particles=["N(1700)+", "Sigma(1660)"],
    formalism="canonical-helicity",
)
REACTION = normalize_state_ids(REACTION)
DECAY = to_three_body_decay(REACTION.transitions, min_ls=True)
Markdown(as_markdown_table([DECAY.initial_state, *DECAY.final_state.values()]))

In [ ]:
Latex(aslatex(DECAY, with_jp=True))

## Model formulation

The unpolarized intensity is an incoherent sum of the aligned amplitude over all helicities, including the helicity $\lambda_0$ of the initial state:

In [ ]:
model_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
for chain in model_builder.decay.chains:
    model_builder.dynamics_choices.register_builder(
        chain, formulate_breit_wigner_with_form_factor
    )
unpolarized_model = model_builder.formulate(reference_subsystem=2)
unpolarized_model.intensity

With {meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate`'s `polarized=True` argument (see {meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate_polarized_intensity`), the incoherent sum over $\lambda_0$ is replaced by

$$
I = \sum_{\mu,\mu'} \rho_{\mu,\mu'} \sum_{\nu,\nu'} \sum_{\{\lambda\}} D^{j_0*}_{\mu,\nu}\left(\phi, \theta, \chi\right) D^{j_0}_{\mu',\nu'}\left(\phi, \theta, \chi\right) A_{\nu,\{\lambda\}} A^*_{\nu',\{\lambda\}}\,,
$$

where $A_{\nu,\{\lambda\}}$ is the aligned amplitude, $\mu, \mu'$ are the spin projections of the initial state onto the quantization axis of the production frame, and $\rho$ is the spin-density matrix of the initial state, represented by a {class}`~sympy.matrices.expressions.MatrixSymbol`. Its rows and columns are ordered from highest to lowest spin projection, so the matrix elements appear with indices $(j_0-\mu,\, j_0-\mu')$. Note also that the complex conjugate of the first Wigner-$D$ function is formulated as $D^{j_0*}_{\mu,\nu}(\phi,\theta,\chi) = D^{j_0}_{\mu,\nu}(-\phi,\theta,-\chi)$.

In [ ]:
model = model_builder.formulate(reference_subsystem=2, polarized=True)
model.intensity

The angles are defined in the same way as in the unpolarized case&mdash;the Euler angles $(\phi, \theta, \chi)$ are new, independent kinematic variables (see {obj}`~ampform_dpd.polarization.EULER_ANGLES`).

In [ ]:
Latex(aslatex(model.variables))

## Spin-density matrices

The {mod}`ampform_dpd.polarization` module provides a few common parametrizations of the spin-density matrix. An unpolarized initial state corresponds to $\rho = \tfrac{1}{2j_0+1}\,\mathbb{1}$. A $J/\psi$ produced through unpolarized $e^+e^-$ annihilation can only have spin projections $m=\pm1$ along the beam axis, without coherence between them:

In [ ]:
ρ = create_spin_density_matrix(DECAY.initial_state.spin)
ρ_unpolarized = formulate_unpolarized_density(DECAY.initial_state.spin)
ρ_ee = formulate_ee_annihilation_density()
Latex(
    aslatex({
        sp.Symbol(R"\rho_\mathrm{unpolarized}"): ρ_unpolarized,
        sp.Symbol(R"\rho_{e^+e^-}"): ρ_ee,
    })
)

For a spin-½ initial state, such as $\Lambda_c^+$ in {doc}`lc2pkpi`, the density matrix can be parametrized with a polarization vector $\vec{P} = (P_x, P_y, P_z)$:

In [ ]:
P = sp.symbols("P_x P_y P_z", real=True)
Latex(
    aslatex({
        create_spin_density_matrix(spin=0.5): formulate_polarized_spin_half_density(*P)
    })
)

## Preparing for input data

Since $\rho$ is a {class}`~sympy.matrices.expressions.MatrixSymbol`, a specific polarization scenario can be selected by substituting it with an explicit {class}`~sympy.matrices.immutable.ImmutableDenseMatrix`. Here, we create two numerical functions from the same polarized model: one with the unpolarized density matrix and one with the $e^+e^-$ density matrix.

In [ ]:
full_intensity_expr = cached.unfold(model)
intensity_funcs = {
    label: cached.lambdify(
        cached.xreplace(
            full_intensity_expr.xreplace({ρ: density_matrix}),
            model.parameter_defaults,
        ),
        backend="jax",
    )
    for label, density_matrix in {
        "unpolarized": ρ_unpolarized,
        R"$e^+e^-$ polarization": ρ_ee,
    }.items()
}
{label: func.argument_order for label, func in intensity_funcs.items()}

Both density matrices are diagonal, so the dependence on $\phi$ drops out of the expressions. With the unpolarized density matrix, the intensity does not depend on the remaining Euler angles either&mdash;the Wigner-$D$ functions sum up to unity (unitarity)&mdash;as the consistency check below shows numerically.

We generate a uniform phase space sample over the Dalitz variables $(\sigma_1, \sigma_2)$ and the orientation angles. Points outside the physical region are filtered out with the unpolarized intensity.

In [ ]:
definitions = {
    symbol: expr.xreplace(model.masses) for symbol, expr in model.variables.items()
}
transformer = SympyDataTransformer.from_sympy(definitions, backend="jax")
rng = np.random.default_rng(seed=0)
n_events = 500_000
m0, m1, m2, m3 = (float(v) for v in model.masses.values())
phsp = {
    "sigma1": rng.uniform((m2 + m3) ** 2, (m0 - m1) ** 2, n_events),
    "sigma2": rng.uniform((m1 + m3) ** 2, (m0 - m2) ** 2, n_events),
    "phi": rng.uniform(-np.pi, +np.pi, n_events),
    "theta": np.arccos(rng.uniform(-1, +1, n_events)),
    "chi": rng.uniform(-np.pi, +np.pi, n_events),
}
phsp["sigma3"] = m0**2 + m1**2 + m2**2 + m3**2 - phsp["sigma1"] - phsp["sigma2"]
phsp.update(transformer(phsp))
intensities = {label: jnp.real(func(phsp)) for label, func in intensity_funcs.items()}
selector = jnp.isfinite(intensities["unpolarized"])
phsp = {k: array[selector] if jnp.ndim(array) else array for k, array in phsp.items()}
intensities = {label: array[selector] for label, array in intensities.items()}
{label: array.shape for label, array in intensities.items()}

As a consistency check, the polarized intensity with the unpolarized density matrix is exactly the unpolarized model divided by the number of spin projections, $2j_0+1$, for *any* orientation of the decay plane:

In [ ]:
unpolarized_intensity_func = cached.lambdify(
    cached.xreplace(cached.unfold(unpolarized_model), model.parameter_defaults),
    backend="jax",
)
n_projections = int(2 * DECAY.initial_state.spin + 1)
np.testing.assert_allclose(
    n_projections * np.asarray(intensities["unpolarized"]),
    np.asarray(unpolarized_intensity_func(phsp)),
)

## Effect of polarization

The polarization of the initial state becomes visible in the distribution of the orientation angles. For the $e^+e^-$ density matrix, the distribution remains flat in $\phi$, but is modulated in $\cos\theta$ and $\chi$:

In [ ]:
%config InlineBackend.figure_formats = ['svg']
plt.rc("font", size=12)
angular_variables = {
    "phi": (phsp["phi"], R"$\phi$"),
    "theta": (np.cos(phsp["theta"]), R"$\cos\theta$"),
    "chi": (phsp["chi"], R"$\chi$"),
}
fig, axes = plt.subplots(figsize=(12, 4), ncols=3, layout="constrained")
for ax, (x, x_label) in zip(axes, angular_variables.values(), strict=True):
    for label, weights in intensities.items():
        bin_values, bin_edges = np.histogram(x, bins=50, weights=weights, density=True)
        ax.stairs(bin_values, bin_edges, label=label, lw=2)
    ax.set_xlabel(x_label)
    ax.set_ylim(0, None)
axes[0].set_ylabel("Normalized intensity (a.u.)")
axes[-1].legend(fontsize=10)
plt.show()

The modulations in $\cos\theta$ and $\chi$ are correlated. This becomes visible in the two-dimensional distribution of the intensity *ratio* between the polarized and the unpolarized case, which divides out the structure of the Dalitz plane:

In [ ]:
ratio = intensities[R"$e^+e^-$ polarization"] / intensities["unpolarized"]
x, y = phsp["chi"], np.cos(phsp["theta"])
bins = 50
summed_ratio, x_edges, y_edges = np.histogram2d(x, y, bins=bins, weights=ratio)
counts, *_ = np.histogram2d(x, y, bins=bins)
mean_ratio = summed_ratio / counts
fig, ax = plt.subplots(figsize=(8, 5))
mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    mean_ratio.T,
    cmap="RdBu_r",
    norm=CenteredNorm(vcenter=1),
    rasterized=True,
)
c_bar = fig.colorbar(mesh, ax=ax, pad=0.02)
c_bar.ax.set_ylabel("Mean intensity ratio (polarized / unpolarized)")
ax.set_xlabel(R"$\chi$")
ax.set_ylabel(R"$\cos\theta$")
plt.show()

The distribution over the Dalitz plane itself is *not* affected by the polarization: integrating the polarized intensity over the orientation angles gives back the unpolarized Dalitz distribution for any choice of $\rho$ (orthogonality of the Wigner-$D$ functions). Indeed, the distributions of the Mandelstam variables coincide:

In [ ]:
fig, axes = plt.subplots(figsize=(12, 4), ncols=2, sharey=True)
fig.subplots_adjust(wspace=0.04)
for ax, (sigma_key, x_label) in zip(
    axes,
    {
        "sigma2": R"$\sigma_2 = M^2\left(K^0 \overline{p}\right)$",
        "sigma3": R"$\sigma_3 = M^2\left(K^0 \Sigma^+\right)$",
    }.items(),
    strict=True,
):
    for label, weights in intensities.items():
        bin_values, bin_edges = np.histogram(
            phsp[sigma_key], bins=80, weights=weights, density=True
        )
        ax.stairs(bin_values, bin_edges, label=label, lw=2)
    ax.set_xlabel(x_label)
    ax.set_ylim(0, None)
axes[0].set_ylabel("Normalized intensity (a.u.)")
axes[-1].legend(fontsize=10)
plt.show()

## A polarized baryon: $\Lambda_c^+ \to p K^- \pi^+$

For a spin-½ initial state, the spin-density matrix is fully parametrized by a polarization vector $\vec{P}$ (see the matrix at the end of [the previous section](#spin-density-matrices)). LHCb measured the polarization of $\Lambda_c^+$ baryons produced in semileptonic beauty-hadron decays through an amplitude analysis of $\Lambda_c^+ \to pK^-\pi^+$ and found $\vec{P} \approx (21.7\%, 1.1\%, -66.5\%)$, that is, a polarization of about 70% {cite}`LHCb:2022sck`. We build a small model with two resonances for this decay:

In [ ]:
LC_REACTION = qrules.generate_transitions(
    initial_state="Lambda(c)+",
    final_state=["p", "K-", "pi+"],
    allowed_intermediate_particles=["Lambda(1520)", "Delta(1232)++"],
    formalism="canonical-helicity",
)
LC_REACTION = normalize_state_ids(LC_REACTION)
LC_DECAY = to_three_body_decay(LC_REACTION.transitions, min_ls=True)
lc_builder = DalitzPlotDecompositionBuilder(LC_DECAY, min_ls=True)
for chain in lc_builder.decay.chains:
    lc_builder.dynamics_choices.register_builder(
        chain, formulate_breit_wigner_with_form_factor
    )
lc_model = lc_builder.formulate(polarized=True)
Latex(aslatex(LC_DECAY, with_jp=True))

The measured polarization vector corresponds to the density matrix

In [ ]:
P_lhcb = (0.2165, 0.0108, -0.665)
ρ_half = create_spin_density_matrix(spin=0.5)
ρ_lhcb = formulate_polarized_spin_half_density(*P_lhcb)
Latex(aslatex({ρ_half: ρ_lhcb}))

The polarization only becomes visible in the angular distributions if the helicity couplings are non-trivial&mdash;with all couplings equal (the default values), this model happens to be insensitive to $\vec{P}$. We therefore draw random complex values for the couplings, as would result from a fit to data, and compare the angular distributions with the unpolarized case:

In [ ]:
lc_rng = np.random.default_rng(seed=3)
lc_parameters = dict(lc_model.parameter_defaults)
for symbol in lc_parameters:
    if isinstance(symbol, sp.Indexed):
        lc_parameters[symbol] = complex(lc_rng.normal(), lc_rng.normal())
lc_intensity_expr = cached.unfold(lc_model)
lc_intensity_funcs = {
    label: cached.lambdify(
        cached.xreplace(lc_intensity_expr.xreplace({ρ_half: density}), lc_parameters),
        backend="jax",
    )
    for label, density in {
        "unpolarized": formulate_unpolarized_density(spin=0.5),
        "LHCb polarization": ρ_lhcb,
    }.items()
}
lc_definitions = {
    symbol: expr.xreplace(lc_model.masses)
    for symbol, expr in lc_model.variables.items()
}
lc_transformer = SympyDataTransformer.from_sympy(lc_definitions, backend="jax")
lc_data_rng = np.random.default_rng(seed=0)
lc_m0, lc_m1, lc_m2, lc_m3 = (float(v) for v in lc_model.masses.values())
lc_phsp = {
    "sigma1": lc_data_rng.uniform((lc_m2 + lc_m3) ** 2, (lc_m0 - lc_m1) ** 2, n_events),
    "sigma2": lc_data_rng.uniform((lc_m1 + lc_m3) ** 2, (lc_m0 - lc_m2) ** 2, n_events),
    "phi": lc_data_rng.uniform(-np.pi, +np.pi, n_events),
    "theta": np.arccos(lc_data_rng.uniform(-1, +1, n_events)),
    "chi": lc_data_rng.uniform(-np.pi, +np.pi, n_events),
}
lc_phsp["sigma3"] = (
    lc_m0**2 + lc_m1**2 + lc_m2**2 + lc_m3**2 - lc_phsp["sigma1"] - lc_phsp["sigma2"]
)
lc_phsp.update(lc_transformer(lc_phsp))
lc_intensities = {
    label: jnp.real(func(lc_phsp)) for label, func in lc_intensity_funcs.items()
}
lc_selector = jnp.isfinite(lc_intensities["unpolarized"])
lc_phsp = {k: v[lc_selector] if jnp.ndim(v) else v for k, v in lc_phsp.items()}
lc_intensities = {label: v[lc_selector] for label, v in lc_intensities.items()}
lc_angular_variables = {
    "phi": (lc_phsp["phi"], R"$\phi$"),
    "theta": (np.cos(lc_phsp["theta"]), R"$\cos\theta$"),
    "chi": (lc_phsp["chi"], R"$\chi$"),
}
fig, axes = plt.subplots(figsize=(12, 4), ncols=3, layout="constrained")
for ax, (x, x_label) in zip(axes, lc_angular_variables.values(), strict=True):
    for label, weights in lc_intensities.items():
        bin_values, bin_edges = np.histogram(x, bins=50, weights=weights, density=True)
        ax.stairs(bin_values, bin_edges, label=label, lw=2)
    ax.set_xlabel(x_label)
    ax.set_ylim(0, None)
axes[0].set_ylabel("Normalized intensity (a.u.)")
axes[-1].legend(fontsize=10)
plt.show()

:::{seealso}
{meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate_polarized_intensity` and the {mod}`ampform_dpd.polarization` module for more information about the implementation.
:::